# WO-03 Q3 — ecoregions
How many ecoregions (`gaz."Ecoregions2017"`, 847) intersect a core basin and its ring, with area shares; whether the ecoregion containing the place's point is the core's largest-share ecoregion; the distribution over the WO-02 sample; and which ecoregions have no Wikipedia link or summary (`public.eco_wikitext`). 69 ecoregion geometries are invalid, so intersections use `ST_MakeValid` for those. Needs `output/edop/whg/q1_pairs.csv` (Q1 notebook, Cell 12). Read-only.

In [1]:
%matplotlib inline
# Cell 1 — Setup; ecoregion metadata and the 69 invalid geometries
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
ECO = pd.DataFrame(conn.execute('SELECT eco_id::bigint, eco_name, biome_name, realm FROM gaz."Ecoregions2017"').fetchall(),
                   columns=['eco_id', 'eco_name', 'biome_name', 'realm']).set_index('eco_id')
INVALID = [int(r[0]) for r in conn.execute('SELECT eco_id FROM gaz."Ecoregions2017" WHERE NOT ST_IsValid(geom)').fetchall()]
print('commit', C.commit(), '| ecoregions:', len(ECO), '| invalid geometries:', len(INVALID), '| pairs:', len(PAIRS))

commit 6f70304 | ecoregions: 847 | invalid geometries: 69 | pairs: 1739


In [2]:
# Cell 2 — Wikipedia link and LLM summary coverage over all ecoregions
W = pd.DataFrame(conn.execute("SELECT eco_id::bigint, wiki_url IS NOT NULL AND wiki_url <> '', summary IS NOT NULL AND summary <> '' FROM public.eco_wikitext").fetchall(),
                 columns=['eco_id', 'has_link', 'has_summary']).set_index('eco_id')
ECO['in_wikitext'] = ECO.index.isin(W.index)
ECO['has_link'] = W['has_link'].reindex(ECO.index).fillna(False).astype(bool)
ECO['has_summary'] = W['has_summary'].reindex(ECO.index).fillna(False).astype(bool)
miss = ECO[~(ECO['has_link'] & ECO['has_summary'])]
L = [f"ecoregions: {len(ECO)} | in eco_wikitext: {int(ECO['in_wikitext'].sum())} | with link: {int(ECO['has_link'].sum())} | with summary: {int(ECO['has_summary'].sum())} | missing either: {len(miss)}",
     f"eco_wikitext rows with no matching ecoregion: {int((~W.index.isin(ECO.index)).sum())}", '',
     'missing by realm:', miss['realm'].value_counts().to_string(), '',
     'the ecoregions without a link or summary:', miss[['eco_name', 'biome_name', 'realm']].to_string()]
print('\n'.join(L))

/var/folders/f9/r5mr431d23zcpktsjz_xjxt40000gn/T/ipykernel_88481/1673560391.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ECO['has_link'] = W['has_link'].reindex(ECO.index).fillna(False).astype(bool)
/var/folders/f9/r5mr431d23zcpktsjz_xjxt40000gn/T/ipykernel_88481/1673560391.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ECO['has_summary'] = W['has_summary'].reindex(ECO.index).fillna(False).astype(bool)


In [3]:
# Cell 3 — Ecoregion area shares of one basin (function) and a timing test on the nine test cores
SQL = """
SELECT e.eco_id::bigint, ST_Area(ST_Intersection(b.geom, ev.g)::geography) / 1e6 AS km2, ST_Area(b.geom::geography) / 1e6 AS basin_km2
FROM public.basin06 b
JOIN gaz."Ecoregions2017" e ON e.geom && b.geom
CROSS JOIN LATERAL (SELECT CASE WHEN e.eco_id = ANY(%s) THEN ST_MakeValid(e.geom) ELSE e.geom END AS g) ev
WHERE b.hybas_id = %s AND ST_Intersects(ev.g, b.geom)
"""
def eco_shares(hid):
    r = pd.DataFrame(conn.execute(SQL, (INVALID, int(hid))).fetchall(), columns=['eco_id', 'km2', 'basin_km2'])
    return r
test_cores = PAIRS[PAIRS.source == 'test'].drop_duplicates('core_id')[['tag', 'core_id']]
rows = []
for r in test_cores.itertuples():
    t0 = time.perf_counter(); d = eco_shares(r.core_id); dt = time.perf_counter() - t0
    pos = d[d['km2'] > 0]
    rows.append(dict(place=r.tag, core_id=r.core_id, seconds=round(dt, 2), n_intersect=len(d), n_with_area=len(pos),
                     covered_share=round(pos['km2'].sum() / d['basin_km2'].iloc[0], 3) if len(d) else None))
print(pd.DataFrame(rows).to_string(index=False))

       place    core_id  seconds  n_intersect  n_with_area  covered_share
    Timbuktu 1060551560     0.10            2            2          1.000
     Tbilisi 2060616700     0.03            3            3          1.000
    Santa Fe 7060674070     0.29            5            5          1.000
     Kaifeng 4060675920     0.03            1            1          1.000
        Lviv 2060376210     0.03            1            1          1.000
     Poltava 2060459130     0.03            1            1          1.000
Chaco Canyon 7060618200     0.02            2            2          1.000
  Alexandria 1060034170     0.12            3            3          0.993
   Utqiagvik 8060007840     0.03            1            1          0.973


In [4]:
# Cell 4 — Every core and neighbour in the pair table (about 2,000 basins): ecoregion areas, saved to a CSV; resumes where it stopped
# Set MAX_BASINS to a number to do a partial run; run the cell again to continue.
MAX_BASINS = None
OUTCSV = C.OUT / 'q3_basin_eco.csv'
ids = sorted(set(PAIRS['core_id']) | set(PAIRS['nb_id']))
done = pd.read_csv(OUTCSV) if OUTCSV.exists() else pd.DataFrame(columns=['basin_id', 'eco_id', 'km2', 'basin_km2'])
done_ids = set(done['basin_id'].astype('int64'))
todo = [i for i in ids if i not in done_ids]
if MAX_BASINS is not None:
    todo = todo[:MAX_BASINS]
print(f'basins: {len(ids)} | already done: {len(done_ids)} | this run: {len(todo)}')
new, t_start = [], time.perf_counter()
for k, hid in enumerate(todo, 1):
    d = eco_shares(hid); d['basin_id'] = hid
    if d.empty:
        d = pd.DataFrame([dict(eco_id=-1, km2=0.0, basin_km2=np.nan, basin_id=hid)])
    new.append(d)
    if k % 200 == 0:
        print(f'  {k} of {len(todo)} in {time.perf_counter() - t_start:.0f}s')
if new:
    done = pd.concat([done, pd.concat(new)[['basin_id', 'eco_id', 'km2', 'basin_km2']]], ignore_index=True)
    done.to_csv(OUTCSV, index=False)
print(f'rows saved: {len(done)} | basins covered: {done["basin_id"].nunique()} of {len(ids)} | this run took {time.perf_counter() - t_start:.0f}s')

/var/folders/f9/r5mr431d23zcpktsjz_xjxt40000gn/T/ipykernel_88481/3322495661.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  done = pd.concat([done, pd.concat(new)[['basin_id', 'eco_id', 'km2', 'basin_km2']]], ignore_index=True)


In [5]:
# Cell 5 — Ecoregions per core and per ring: counts, shares, and the place's own ecoregion
EB = pd.read_csv(C.OUT / 'q3_basin_eco.csv')
EB = EB[EB['eco_id'] >= 0]
print('basins with no ecoregion row at all:', len(set(PAIRS['core_id']) | set(PAIRS['nb_id'])) - EB['basin_id'].nunique())
bas_area = EB.drop_duplicates('basin_id').set_index('basin_id')['basin_km2']
def shares(basin_ids):
    sub = EB[(EB['basin_id'].isin(basin_ids)) & (EB['km2'] > 0)]
    tot = bas_area.reindex(list(basin_ids)).sum()
    g = sub.groupby('eco_id')['km2'].sum().sort_values(ascending=False)
    return g / tot
def thr(s, t):
    return int((s >= t).sum())
def pt_eco(lat, lon):
    r = conn.execute("""
        SELECT e.eco_id FROM gaz."Ecoregions2017" e
        CROSS JOIN LATERAL (SELECT CASE WHEN e.eco_id = ANY(%s) THEN ST_MakeValid(e.geom) ELSE e.geom END AS g) ev
        WHERE e.geom && ST_SetSRID(ST_MakePoint(%s, %s), 4326) AND ST_Intersects(ev.g, ST_SetSRID(ST_MakePoint(%s, %s), 4326))""",
        (INVALID, lon, lat, lon, lat)).fetchall()
    return [int(x[0]) for x in r]
sample_pt = {int(p['hybas_id_l6']): (p['lat'], p['lon']) for p in C.population_points()}
rows = []
for core, grp in PAIRS.groupby('core_id'):
    src = grp['source'].iloc[0]; tag = grp['tag'].iloc[0]
    lat, lon = C.PLACES[tag] if src == 'test' else sample_pt[int(tag)]
    sc = shares([core]); sr = shares([core] + list(grp['nb_id']))
    pe = pt_eco(lat, lon)
    top = int(sc.index[0]) if len(sc) else None
    rows.append(dict(source=src, tag=tag, core_id=core, n_core=len(sc), n_core_1pct=thr(sc, .01), n_core_5pct=thr(sc, .05),
                     n_ring=len(sr), n_ring_1pct=thr(sr, .01), n_ring_5pct=thr(sr, .05),
                     core_top_share=round(float(sc.iloc[0]), 3) if len(sc) else None, core_covered=round(float(sc.sum()), 3), ring_covered=round(float(sr.sum()), 3),
                     point_eco=pe[0] if len(pe) == 1 else (None if not pe else -2), n_point_matches=len(pe),
                     point_eq_top=(pe[0] == top) if len(pe) == 1 else None, core_top=top))
RES = pd.DataFrame(rows)
RES.to_csv(C.OUT / 'q3_core_ring_eco.csv', index=False)
print('saved q3_core_ring_eco.csv;', len(RES), 'cores')
t = RES[RES.source == 'test'].copy()
t['point_eco_name'] = t['point_eco'].map(ECO['eco_name']); t['core_top_name'] = t['core_top'].map(ECO['eco_name'])
print(t[['tag', 'n_core', 'n_core_5pct', 'n_ring', 'n_ring_5pct', 'core_top_share', 'point_eco_name', 'core_top_name', 'point_eq_top']].to_string(index=False))

saved q3_core_ring_eco.csv; 307 cores
         tag  n_core  n_core_5pct  n_ring  n_ring_5pct  core_top_share                         point_eco_name                           core_top_name point_eq_top
  Alexandria       3            3       5            2           0.688 Mediterranean dry woodlands and steppe North Saharan Xeric Steppe and Woodland        False
    Timbuktu       2            2       2            2           0.687                Sahelian Acacia savanna       Inner Niger Delta flooded savanna        False
        Lviv       1            1       2            2           1.000         Central European mixed forests          Central European mixed forests         True
     Poltava       1            1       3            2           1.000            East European forest steppe             East European forest steppe         True
     Tbilisi       3            3       7            3           0.787     Azerbaijan shrub desert and steppe                  Caucasus mixed fores

In [6]:
# Cell 6 — The sample: distribution of the count, and does the point's ecoregion equal the core's largest-share ecoregion?
S = RES[RES.source == 'sample']
L = [f'sample cores: {len(S)}']
for col, lab in [('n_core', 'core: ecoregions with any area'), ('n_core_1pct', 'core: with at least 1% of the core'), ('n_core_5pct', 'core: with at least 5%'),
                 ('n_ring', 'ring: any area'), ('n_ring_1pct', 'ring: at least 1% of the ring'), ('n_ring_5pct', 'ring: at least 5%')]:
    L.append(f'{lab}: ' + S[col].describe(percentiles=[.1, .5, .9]).round(1).to_dict().__repr__())
L.append('')
L.append('core: number of ecoregions with at least 5% of the core (cores):')
L.append(S['n_core_5pct'].value_counts().sort_index().to_string())
L.append('')
L.append('ring: number of ecoregions with at least 5% of the ring (rings):')
L.append(S['n_ring_5pct'].value_counts().sort_index().to_string())
L.append('')
ok = S[S['point_eq_top'].notna()]
L.append(f'point inside exactly one ecoregion: {len(ok)} of {len(S)} (no match: {int((S["n_point_matches"] == 0).sum())}, more than one: {int((S["n_point_matches"] > 1).sum())})')
L.append(f'point ecoregion equals the core\'s largest-share ecoregion: {int(ok["point_eq_top"].sum())} of {len(ok)} ({100 * ok["point_eq_top"].mean():.1f}%)')
L.append('largest-share ecoregion\'s share of the core (all sample cores): ' + S['core_top_share'].describe(percentiles=[.1, .5, .9]).round(2).to_dict().__repr__())
L.append('')
cov = RES[['core_covered', 'ring_covered']].describe(percentiles=[.1, .5]).round(3)
L.append('share of the core / ring covered by any ecoregion (the rest has no ecoregion polygon):')
L.append(cov.to_string())
L.append('')
# coverage of summaries: the point ecoregion and the core's largest ecoregion
for col, lab in [('point_eco', "the point's ecoregion"), ('core_top', "the core's largest-share ecoregion")]:
    s = RES[col].dropna(); s = s[s >= 0].astype(int)
    L.append(f'{lab} has a Wikipedia link and summary: {int(ECO["has_summary"].reindex(s).sum())} of {len(s)} cores')
print('\n'.join(L))

sample cores: 298
core: ecoregions with any area: {'count': 298.0, 'mean': 1.9, 'std': 1.0, 'min': 1.0, '10%': 1.0, '50%': 2.0, '90%': 3.0, 'max': 7.0}
core: with at least 1% of the core: {'count': 298.0, 'mean': 1.8, 'std': 0.9, 'min': 1.0, '10%': 1.0, '50%': 2.0, '90%': 3.0, 'max': 6.0}
core: with at least 5%: {'count': 298.0, 'mean': 1.7, 'std': 0.8, 'min': 1.0, '10%': 1.0, '50%': 2.0, '90%': 3.0, 'max': 5.0}
ring: any area: {'count': 298.0, 'mean': 3.1, 'std': 1.7, 'min': 1.0, '10%': 1.0, '50%': 3.0, '90%': 5.0, 'max': 9.0}
ring: at least 1% of the ring: {'count': 298.0, 'mean': 2.8, 'std': 1.5, 'min': 1.0, '10%': 1.0, '50%': 2.0, '90%': 5.0, 'max': 8.0}
ring: at least 5%: {'count': 298.0, 'mean': 2.4, 'std': 1.2, 'min': 1.0, '10%': 1.0, '50%': 2.0, '90%': 4.0, 'max': 7.0}

core: number of ecoregions with at least 5% of the core (cores):
n_core_5pct
1    146
2    111
3     35
4      5
5      1

ring: number of ecoregions with at least 5% of the ring (rings):
n_ring_5pct
1     76
2 

In [7]:
# Cell 7 — Cell 2 repeated without the reindex/fillna that raised the FutureWarning (same output; run it any time after Cell 2)
ECO['in_wikitext'] = ECO.index.isin(W.index)
ECO['has_link'] = ECO.index.isin(W.index[W['has_link'].to_numpy()])
ECO['has_summary'] = ECO.index.isin(W.index[W['has_summary'].to_numpy()])
miss = ECO[~(ECO['has_link'] & ECO['has_summary'])]
L = [f"ecoregions: {len(ECO)} | in eco_wikitext: {int(ECO['in_wikitext'].sum())} | with link: {int(ECO['has_link'].sum())} | with summary: {int(ECO['has_summary'].sum())} | missing either: {len(miss)}",
     f"eco_wikitext rows with no matching ecoregion: {int((~W.index.isin(ECO.index)).sum())}", '',
     'missing by realm:', miss['realm'].value_counts().to_string(), '',
     'the ecoregions without a link or summary:', miss[['eco_name', 'biome_name', 'realm']].to_string()]
print('\n'.join(L))

ecoregions: 847 | in eco_wikitext: 821 | with link: 821 | with summary: 821 | missing either: 26
eco_wikitext rows with no matching ecoregion: 0

missing by realm:
realm
Nearctic       10
Afrotropic      6
Antarctica      5
Palearctic      3
Neotropic       1
Australasia     1

the ecoregions without a link or summary:
                                                eco_name                                                biome_name        realm
eco_id                                                                                                                         
365              Queen Charlotte Islands conifer forests                                 Temperate Conifer Forests     Nearctic
400            Southeast US mixed woodlands and savannas               Temperate Grasslands, Savannas & Shrublands     Nearctic
490                        Pantepui forests & shrublands            Tropical & Subtropical Moist Broadleaf Forests    Neotropic
404                     Ahklun and Kilb